# Choose a distribution for SPEI

This notebook runs the distribution-suitability workflow on the prepared
nClimGrid low-resolution sample. It answers a practical question: which
distribution should SPEI use at Timescales 3, 6, 12 and 24 months, and does the
answer change the drought categories and events a user sees?

It is a **workflow reproduction, not a reference reproduction**. It does not
reproduce Monish & Rehana (2020), the study that motivates these tools. That
study is cited by DOI only (`10.1007/s12040-019-1271-x`), its data and verified
methods are not available here, and this notebook makes no claim about its
numbers.

## Data and how to run this notebook

From the repository root, prepare the pinned sample once:

```bash
uv run --group dev scripts/prepare_e2e_inputs.py
```

The script downloads about 15 MB and writes the git-ignored store under
`data/e2e/`. Start the kernel in `notebooks/` so the default `../data/e2e` path
resolves, or set `CLIMATE_INDICES_E2E_DATA` to the prepared-data directory. The
notebook downloads no source data itself.

## Outline

1. Compare gamma, Pearson Type III, log-logistic and GEV by AICc at four
   Timescales.
2. Check the goodness of fit, first with naive p-values and then with a
   parametric bootstrap.
3. Measure how often the top two candidates disagree on the USDM drought
   category.
4. Compare the seasonal PIT of SPEI against SPAI at a representative cell.
5. Count the agreement of SPEI events with a labelled demonstration reference.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import climate_indices.suitability as suitability
from climate_indices import spai, spei
from climate_indices.runs import identify_runs
from climate_indices.vocabulary import Distribution

SCALES = (3, 6, 12, 24)
CALIBRATION = {"calibration_year_initial": 1981, "calibration_year_final": 2010}

data_root = Path(os.environ.get("CLIMATE_INDICES_E2E_DATA", "../data/e2e"))
prepared = data_root / "current" / "cache_prepared_input.zarr"
if not prepared.exists():
    raise FileNotFoundError(
        f"{prepared} missing; run uv run --group dev scripts/prepare_e2e_inputs.py"
    )

ds = xr.open_zarr(prepared, consolidated=True)
precip = ds["precip"].load()
pet = ds["pet"].load()
wb = ds["wb"].load()
if precip.attrs.get("units") != "mm" or pet.attrs.get("units") != "mm":
    raise ValueError("Expected monthly precipitation and PET totals in mm")

land = precip.notnull().any("time").values
row, col = (int(index) for index in np.argwhere(land)[0])
cell = {"lat": float(precip["lat"].values[row]), "lon": float(precip["lon"].values[col])}
print(f"Representative cell row/column {row}/{col} at lat {cell['lat']:.2f}, lon {cell['lon']:.2f}")
print(f"{int(land.sum())} land cells of {land.size}; {precip.sizes['time']} months")


## 1. Relative fit: which candidate wins on AICc?

`compare_distributions()` prepares the SPEI water balance (`P - PET + 1000`) and
fits every candidate on one shared calibration sample, 1981-2010. The default
criterion is AICc. The library fits with its own estimators, Thom's closed-form
gamma approximation and L-moments, so the criteria are **penalized likelihoods
at fitted estimates**, not textbook AIC/AICc/BIC. A ranking mixes how well a
family fits with how efficient its estimator is.

`best` is the argmin of the criterion, with no significance threshold, so read
`delta_to_runner_up` next to it. About 30 values per calendar month discriminate
weakly, and a small delta does not separate two candidates. A Timescale above 12
months makes consecutive years' windows overlap, so the call warns and the
goodness-of-fit p-values are invalid there. A failed fit is recorded in
`fit_failed`; it is never replaced by another family.


In [ ]:
comparisons = {
    scale: suitability.compare_distributions(
        precip, scale, index="spei", pet_mm=pet, **CALIBRATION, gof=("ks", "ad")
    )
    for scale in SCALES
}
candidates = [str(label) for label in comparisons[SCALES[0]]["candidate"].values]
print("Candidates:", candidates)


In [ ]:
def best_shares(comparison, candidates):
    values = comparison["best"].values
    shares = {label: int(np.count_nonzero(values == label)) / values.size for label in candidates}
    shares["none"] = float(np.count_nonzero(values == "none")) / values.size
    return shares


shares = pd.DataFrame(
    {scale: best_shares(comparisons[scale], candidates) for scale in SCALES}
).T
deltas = pd.Series(
    {
        scale: float(np.nanmedian(comparisons[scale]["delta_to_runner_up"].values))
        for scale in SCALES
    }
)
print("Share of best candidate by AICc, over every month and cell:")
print(shares.round(3))
print("\nMedian AICc gap to the runner-up:")
print(deltas.round(2))

axes = shares[[*candidates, "none"]].plot.bar(stacked=True, figsize=(9, 4), rot=0)
axes.set(
    title="Share of best candidate by AICc",
    xlabel="Timescale (months)",
    ylabel="Share of month-cell positions",
)
axes.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()


## 2. Absolute fit: does the chosen family fit?

AICc ranks candidates against each other. It says nothing about whether any
family fits. `compare_distributions(..., gof=("ks", "ad"))` adds absolute
statistics on each candidate's probability integral transform: the
Kolmogorov-Smirnov D with a naive p-value and the Anderson-Darling A^2, which
has no naive p-value.

The parameters are estimated from the same sample the test scores, so the naive
K-S p-value is uncalibrated. The classical Lilliefors result covers
maximum-likelihood estimates, not the estimators used here, so the direction of
the miscalibration is not established (ADR-0028). `n_bootstrap` draws from the
fitted distribution, refits every draw with the library's estimator, and gives
p-values that account for the estimation. The cost is about `n_bootstrap` times
the cost of the fits, so the bootstrap below runs on a small block of land
cells. A calendar step holds about 30 values here, so even the bootstrap
rejection share is noisy and can sit above the nominal level. Use it to compare
candidates, not as a calibrated error rate. A p-value for the candidate that
`best` selected is a post-selection statistic, not a confirmatory test.

The prepared precipitation field is a multiple of 1/1024 mm and PET is
continuous, so the SPEI water balance scored here is not on a fixed lattice and
the continuous draws (`resolution=None`) are the right null. For a record
rounded to a unit, pass that unit as `resolution`. The library never infers it.


In [ ]:
N_BOOTSTRAP = 199
block_size = min(6, precip.sizes["lat"], precip.sizes["lon"])
_, block_row, block_col = max(
    (
        (float(land[r : r + block_size, c : c + block_size].mean()), r, c)
        for r in range(land.shape[0] - block_size + 1)
        for c in range(land.shape[1] - block_size + 1)
    ),
    key=lambda candidate: candidate[0],
)
crop = {
    "lat": slice(block_row, block_row + block_size),
    "lon": slice(block_col, block_col + block_size),
}
boot = suitability.compare_distributions(
    precip.isel(**crop),
    3,
    index="spei",
    pet_mm=pet.isel(**crop),
    **CALIBRATION,
    gof=("ks", "ad"),
    n_bootstrap=N_BOOTSTRAP,
    seed=42,
)
ks_naive = boot["ks_p_value_naive"]
naive = (ks_naive <= 0.05).where(ks_naive.notnull()).mean(dim=["month", "lat", "lon"]).to_pandas()
rejection = pd.DataFrame(
    {
        "KS naive": naive,
        "KS bootstrap": boot["rejection_share"].sel(statistic="ks").mean(dim="month").to_pandas(),
        "AD bootstrap": boot["rejection_share"].sel(statistic="ad").mean(dim="month").to_pandas(),
    }
)
print(f"Bootstrap block of {block_size}x{block_size} cells at row {block_row}, column {block_col}")
print(f"Median shared calibration sample size: {float(np.nanmedian(boot['n'].values)):.0f}")
print(f"Share of positions with a p-value at or below 0.05, over {N_BOOTSTRAP} draws:")
print(rejection.round(3))
print("\nShare of refits that failed during the bootstrap:")
print(boot["bootstrap_failed_share"].mean(dim=["month", "lat", "lon"]).round(4).to_pandas())

axes = rejection.plot.bar(figsize=(9, 4), rot=0)
axes.axhline(0.05, color="black", linestyle="--", linewidth=1)
axes.set(
    title="Goodness-of-fit rejection share against nominal 0.05",
    xlabel="Candidate",
    ylabel="Share of positions with p <= 0.05",
)
plt.tight_layout()
plt.show()


## 3. Decision impact: does the best candidate change the category?

A ranking is abstract; a category change is what a user notices.
`drought_category()` maps normal-scale index values to the U.S. Drought
Monitor classes (None and D0 to D4). `compare_categories()` counts how often two
classifications of the same values disagree. Only the dry side is classified, so
every wet value is "none". `share_changed_in_drought` counts a pair as in
drought when either classification is D0 to D4.


In [ ]:
scale = 3
ranking = shares.loc[scale, candidates].sort_values(ascending=False)
top_two = list(ranking.index[:2])
print("Top two by AICc share at SPEI-3:", top_two)

index_values = {
    label: spei(precip, pet, distribution=Distribution(label), scale=scale, **CALIBRATION)
    for label in top_two
}
categories = {
    label: suitability.drought_category(index_values[label], output_scale="normal")
    for label in top_two
}
change = suitability.compare_categories(categories[top_two[0]], categories[top_two[1]], dims="time")
print(f"Share of month-cell pairs whose category changes: {float(change['share_changed'].mean()):.3f}")
print(
    "Share changed among pairs where at least one side is in drought: "
    f"{float(change['share_changed_in_drought'].mean()):.3f}"
)

change["share_changed"].plot(figsize=(8, 4), cmap="magma", vmin=0, vmax=1)
plt.title(f"Share of months whose category changes: {top_two[0]} with {top_two[1]}")
plt.show()


## 4. Seasonality: SPEI against SPAI

SPEI fits one distribution per calendar month. SPAI removes each calendar
month's calibration mean and fits one pooled distribution per cell, so its
probabilities mix months with different spreads. `pit_uniformity()` tests the
probability-scale output of either index against a uniform distribution at each
calendar step. A low p-value flags a step to inspect. The p-values are in-sample
and naive, so they can be conservative.


In [ ]:
spei_pit = spei(
    precip, pet, distribution=Distribution(top_two[0]), scale=3, **CALIBRATION, output_scale="probability"
)
spai_pit = spai(
    precip, distribution=Distribution.t_location_scale, scale=3, **CALIBRATION, output_scale="probability"
)
spei_seasonal = suitability.pit_uniformity(
    spei_pit.isel(lat=row, lon=col),
    3,
    distribution=Distribution(top_two[0]),
    **CALIBRATION,
)
spai_seasonal = suitability.pit_uniformity(
    spai_pit.isel(lat=row, lon=col),
    3,
    distribution=Distribution.t_location_scale,
    **CALIBRATION,
)

fig, ax = plt.subplots(figsize=(8, 4))
spei_seasonal["ks_p_value_naive"].plot(ax=ax, marker="o", label="SPEI-3, per-month fit")
spai_seasonal["ks_p_value_naive"].plot(ax=ax, marker="s", label="SPAI-3, pooled fit")
ax.axhline(0.05, color="black", linestyle="--", linewidth=1)
ax.set(
    title=f"Seasonal PIT: naive K-S p-values at lat {cell['lat']:.2f}, lon {cell['lon']:.2f}",
    xlabel="Month",
    ylabel="Naive K-S p-value",
)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Events against a labelled demonstration reference

`event_concordance()` counts hits, misses and false alarms of an index's events
against a reference event set. The counting is reference-centric. The reference
must describe the same phenomenon independently for the numbers to mean
anything.

This notebook has no independent dataset, so it builds a **demonstration
reference** from the sample's own water balance: months whose raw `P - PET`
value lies below its 1981-2010 20th percentile. The reference shares its input
with the index, so the counts below demonstrate the workflow and are **not** an
accuracy score. No external oracle is adopted.


In [ ]:
series = index_values[top_two[0]].isel(lat=row, lon=col)
water = wb.isel(lat=row, lon=col)
time = series["time"].values.astype("datetime64[ns]")
valid = np.isfinite(series.values) & np.isfinite(water.values)
threshold = float(water.sel(time=slice("1981", "2010")).quantile(0.2))

index_runs = identify_runs(
    np.where(valid, series.values, np.nan), threshold=-1.0, direction="below", min_duration=2
)
reference_runs = identify_runs(
    np.where(valid, water.values, np.nan), threshold=threshold, direction="below", min_duration=2
)
concordance = suitability.event_concordance(
    index_runs, reference_runs, time=time, valid=valid, cadence="monthly"
)
print(f"Reference threshold, 20th percentile of P - PET: {threshold:.1f} mm")
print(f"Index events: {len(index_runs.start_index)}; reference events: {len(reference_runs.start_index)}")
print(
    f"hits={concordance.hits} misses={concordance.misses} "
    f"false_alarms={concordance.false_alarms}"
)
print(f"POD={concordance.pod:.3f} FAR={concordance.far:.3f} CSI={concordance.csi:.3f}")
print(
    f"Censored index runs: left={concordance.index_left_censored}, "
    f"right={concordance.index_right_censored}"
)


## What this notebook does and does not show

- It shows the workflow on one prepared sample: relative fit, absolute fit,
  category impact, seasonality and event counting.
- It does not reproduce Monish & Rehana (2020) or any published figure. OQ-5 of
  the design note leaves the independent reference open, so the concordance
  numbers use a labelled demonstration reference built from the same sample.
- `best` is evidence, not "correct". Report `delta_to_runner_up`, the
  goodness-of-fit evidence and the category impact together.
- The tools are Beta (ADR-0028), so a minor release can change their interface.

The [distribution-fit guide](../docs/checking-distribution-fit.md) has the full
workflow with its guardrails. `docs/choosing-parameters.md` covers the index
parameters around it.
